In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
from datetime import datetime
import uuid
import traceback

CATALOG = "Stream_Bank_Processing"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

CUSTOMERS_SOURCE = f"{CATALOG}.{SILVER_SCHEMA}.customers"
ACCOUNTS_SOURCE = f"{CATALOG}.{SILVER_SCHEMA}.accounts"
TRANSACTIONS_SOURCE = f"{CATALOG}.{SILVER_SCHEMA}.transactions"

GOLD_DATABASE = f"{CATALOG}.{GOLD_SCHEMA}"

RUN_ID = str(uuid.uuid4())

RUN_START_TIME = datetime.utcnow()

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_DATABASE}")


LOG_TABLE = f"{GOLD_DATABASE}.gold_processing_log"

log_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("records_processed", LongType(), True),
    StructField("error_message", StringType(), True)
])

if not spark.catalog.tableExists(LOG_TABLE):
    (
        spark.createDataFrame([], log_schema)
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(LOG_TABLE)
    )

print("Gold schema and processing log are ready.")
print("Run ID:", RUN_ID)

In [0]:
customers_silver = spark.table(CUSTOMERS_SOURCE)

accounts_silver = spark.table(ACCOUNTS_SOURCE)

transactions_silver = spark.table(TRANSACTIONS_SOURCE)

print("Customers:", customers_silver.count())
print("Accounts:", accounts_silver.count())
print("Transactions:", transactions_silver.count())

In [0]:
customers = customers_silver

if "is_current" in customers.columns:
    customers = customers.filter(
        F.col("is_current") == True
    )

if "is_deleted" in customers.columns:
    customers = customers.filter(
        F.coalesce(F.col("is_deleted"), F.lit(False)) == False
    )

customers = customers.dropDuplicates(["customer_id"])


accounts = accounts_silver

if "is_current" in accounts.columns:
    accounts = accounts.filter(
        F.col("is_current") == True
    )

if "is_deleted" in accounts.columns:
    accounts = accounts.filter(
        F.coalesce(F.col("is_deleted"), F.lit(False)) == False
    )

accounts = accounts.dropDuplicates(["account_id"])


transactions = transactions_silver

transactions = transactions.dropDuplicates(["txn_id"])

print("Business records prepared.")

In [0]:
def append_gold_log(
    table_name,
    start_time,
    end_time,
    status,
    records_processed,
    error_message=None
):
    log_row = [(
        RUN_ID,
        table_name,
        start_time,
        end_time,
        status,
        int(records_processed) if records_processed is not None else 0,
        str(error_message)[:4000] if error_message else None
    )]

    log_df = spark.createDataFrame(log_row, schema=log_schema)

    (
        log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(LOG_TABLE)
    )


def write_gold_table(df, table_name):
    start_time = datetime.utcnow()

    try:
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(f"{GOLD_DATABASE}.{table_name}")
        )

        records_processed = spark.table(
            f"{GOLD_DATABASE}.{table_name}"
        ).count()

        end_time = datetime.utcnow()

        append_gold_log(
            table_name=table_name,
            start_time=start_time,
            end_time=end_time,
            status="SUCCESS",
            records_processed=records_processed
        )

        print(
            f"SUCCESS | {table_name} | "
            f"{records_processed} rows"
        )

        return records_processed

    except Exception as e:

        end_time = datetime.utcnow()

        try:
            append_gold_log(
                table_name=table_name,
                start_time=start_time,
                end_time=end_time,
                status="FAILED",
                records_processed=0,
                error_message=str(e)
            )
        except Exception as log_error:
            print("Logging also failed:", str(log_error))

        print(f"FAILED | {table_name}")
        raise e

In [0]:
customer_age_df = customers.withColumn(
    "customer_age",
    F.floor(
        F.months_between(
            F.current_date(),
            F.to_date("dob")
        ) / 12
    )
)

dim_customers = customer_age_df.select(
    "customer_id",
    "name",
    "gender",
    "city",
    "segment",
    "kyc_status",
    "credit_score",
    "annual_income",
    "join_date",
    "customer_age"
)

dim_customers = (
    dim_customers
    .withColumn("customer_join_year", F.year("join_date"))
    .withColumn("customer_join_month", F.month("join_date"))
)

write_gold_table(dim_customers, "dim_customers")

In [0]:
dim_accounts = accounts.select(
    "account_id",
    "customer_id",
    "account_type",
    "branch",
    "open_date",
    "status",
    "balance",
    "currency"
)

dim_accounts = (
    dim_accounts
    .withColumn(
        "account_open_year",
        F.year("open_date")
    )
    .withColumn(
        "account_open_month",
        F.month("open_date")
    )
    .withColumn(
        "account_age_days",
        F.datediff(
            F.current_date(),
            F.to_date("open_date")
        )
    )
)

write_gold_table(dim_accounts, "dim_accounts")

In [0]:
transaction_dates = transactions.select(
    F.to_date("txn_timestamp").alias("txn_date")
).filter(
    F.col("txn_date").isNotNull()
)

date_bounds = transaction_dates.agg(
    F.min("txn_date").alias("min_date"),
    F.max("txn_date").alias("max_date")
).first()

if date_bounds["min_date"] is not None:

    date_sequence = spark.sql(
        f"""
        SELECT explode(
            sequence(
                DATE('{date_bounds["min_date"]}'),
                DATE('{date_bounds["max_date"]}'),
                INTERVAL 1 DAY
            )
        ) AS date
        """
    )

    dim_date = (
        date_sequence
        .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
        .withColumn("year", F.year("date"))
        .withColumn("quarter", F.quarter("date"))
        .withColumn("month", F.month("date"))
        .withColumn("month_name", F.date_format("date", "MMMM"))
        .withColumn("year_month", F.date_format("date", "yyyy-MM"))
        .withColumn("day", F.dayofmonth("date"))
        .withColumn("day_of_week", F.dayofweek("date"))
        .withColumn("day_name", F.date_format("date", "EEEE"))
        .withColumn("week_of_year", F.weekofyear("date"))
        .withColumn("is_weekend", F.dayofweek("date").isin([1, 7]))
    )

    write_gold_table(dim_date, "dim_date")

else:
    print("No valid transaction dates found.")

In [0]:
txn_base = transactions.select(
    "txn_id",
    "account_id",
    "txn_type",
    "txn_timestamp",
    "channel",
    "merchant_category",
    "amount",
    "status",
    "fraud_flag"
)

txn_base = (
    txn_base
    .withColumn(
        "txn_date",
        F.to_date("txn_timestamp")
    )
    .withColumn(
        "txn_year",
        F.year("txn_timestamp")
    )
    .withColumn(
        "txn_month",
        F.month("txn_timestamp")
    )
    .withColumn(
        "txn_year_month",
        F.date_format("txn_timestamp", "yyyy-MM")
    )
    .withColumn(
        "txn_hour",
        F.hour("txn_timestamp")
    )
    .withColumn(
        "txn_day_of_week",
        F.dayofweek("txn_timestamp")
    )
    .withColumn(
        "amount",
        F.col("amount").cast("double")
    )
    .withColumn(
        "txn_type_normalized",
        F.upper(F.trim(F.col("txn_type")))
    )
    .withColumn(
        "status_normalized",
        F.upper(F.trim(F.col("status")))
    )
    .withColumn(
        "fraud_flag_normalized",
        F.lower(
            F.trim(
                F.col("fraud_flag").cast("string")
            )
        )
    )
)


SUCCESS_STATUSES = [
    "SUCCESS",
    "COMPLETED",
    "SETTLED"
]

FAILED_STATUSES = [
    "FAILED",
    "FAILURE",
    "DECLINED",
    "REJECTED",
    "CANCELLED",
    "CANCELED"
]

txn_base = (
    txn_base
    .withColumn(
        "is_success",
        F.col("status_normalized").isin(SUCCESS_STATUSES)
    )
    .withColumn(
        "is_failed",
        F.col("status_normalized").isin(FAILED_STATUSES)
    )
    .withColumn(
        "is_pending",
        ~F.col("is_success") & ~F.col("is_failed")
    )
    .withColumn(
        "is_fraud",
        F.col("fraud_flag_normalized").isin(
            "true", "1", "yes", "y"
        )
    )
)

# AMOUNT METRICS

txn_base = (
    txn_base
    .withColumn(
        "absolute_amount",
        F.abs(F.col("amount"))
    )
    .withColumn(
        "successful_amount",
        F.when(
            F.col("is_success"),
            F.col("absolute_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "failed_amount",
        F.when(
            F.col("is_failed"),
            F.col("absolute_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "fraud_amount",
        F.when(
            F.col("is_fraud"),
            F.col("absolute_amount")
        ).otherwise(F.lit(0.0))
    )
)

# ACCOUNT AND CUSTOMER ATTRIBUTES

account_attributes = dim_accounts.select(
    "account_id",
    "customer_id",
    "account_type",
    "branch",
    "currency",
    "balance"
)

fact_transactions = (
    txn_base
    .join(
        account_attributes,
        on="account_id",
        how="left"
    )
    .withColumn(
        "date_key",
        F.date_format("txn_date", "yyyyMMdd").cast("int")
    )
    .withColumn(
        "net_amount",
        F.when(
            F.col("txn_type_normalized").isin(
                "CREDIT", "DEPOSIT", "TRANSFER_IN"
            ),
            F.col("absolute_amount")
        )
        .when(
            F.col("txn_type_normalized").isin(
                "DEBIT", "WITHDRAWAL", "TRANSFER_OUT"
            ),
            -F.col("absolute_amount")
        )
        .otherwise(F.lit(None).cast("double"))
    )
)

write_gold_table(
    fact_transactions,
    "fact_transactions"
)

In [0]:
fact = spark.table(
    f"{GOLD_DATABASE}.fact_transactions"
)

gold_transaction_kpis = fact.agg(
    F.count("*").alias("total_transactions"),

    F.countDistinct("customer_id").alias(
        "total_customers"
    ),

    F.countDistinct("account_id").alias(
        "total_accounts"
    ),

    F.sum(
        F.when(F.col("is_success"), 1).otherwise(0)
    ).alias("successful_transactions"),

    F.sum(
        F.when(F.col("is_failed"), 1).otherwise(0)
    ).alias("failed_transactions"),

    F.sum(
        F.when(F.col("is_pending"), 1).otherwise(0)
    ).alias("pending_transactions"),

    F.sum(
        F.when(F.col("is_fraud"), 1).otherwise(0)
    ).alias("fraud_transactions"),

    F.sum("absolute_amount").alias(
        "total_transaction_amount"
    ),

    F.sum("successful_amount").alias(
        "successful_transaction_amount"
    ),

    F.sum("failed_amount").alias(
        "failed_transaction_amount"
    ),

    F.sum("fraud_amount").alias(
        "fraud_transaction_amount"
    ),

    F.avg("absolute_amount").alias(
        "average_transaction_amount"
    ),

    F.max("absolute_amount").alias(
        "maximum_transaction_amount"
    ),

    F.min("absolute_amount").alias(
        "minimum_transaction_amount"
    ),

    F.sum("net_amount").alias(
        "net_transaction_amount"
    )
)

gold_transaction_kpis = (
    gold_transaction_kpis
    .withColumn(
        "success_rate",
        F.when(
            F.col("total_transactions") > 0,
            F.col("successful_transactions")
            / F.col("total_transactions") * 100
        )
    )
    .withColumn(
        "failure_rate",
        F.when(
            F.col("total_transactions") > 0,
            F.col("failed_transactions")
            / F.col("total_transactions") * 100
        )
    )
    .withColumn(
        "fraud_rate",
        F.when(
            F.col("total_transactions") > 0,
            F.col("fraud_transactions")
            / F.col("total_transactions") * 100
        )
    )
)

write_gold_table(
    gold_transaction_kpis,
    "gold_transaction_kpis"
)

In [0]:
gold_daily_transaction_kpis = (
    fact
    .groupBy("txn_date")
    .agg(
        F.count("*").alias("transaction_count"),

        F.countDistinct("customer_id").alias(
            "active_customers"
        ),

        F.countDistinct("account_id").alias(
            "active_accounts"
        ),

        F.sum("absolute_amount").alias(
            "transaction_amount"
        ),

        F.sum("successful_amount").alias(
            "successful_amount"
        ),

        F.sum("failed_amount").alias(
            "failed_amount"
        ),

        F.sum("fraud_amount").alias(
            "fraud_amount"
        ),

        F.sum(
            F.when(F.col("is_success"), 1).otherwise(0)
        ).alias("successful_transactions"),

        F.sum(
            F.when(F.col("is_failed"), 1).otherwise(0)
        ).alias("failed_transactions"),

        F.sum(
            F.when(F.col("is_fraud"), 1).otherwise(0)
        ).alias("fraud_transactions"),

        F.avg("absolute_amount").alias(
            "average_transaction_amount"
        ),

        F.sum("net_amount").alias(
            "net_transaction_amount"
        )
    )
    .withColumn(
        "success_rate",
        F.when(
            F.col("transaction_count") > 0,
            F.col("successful_transactions")
            / F.col("transaction_count") * 100
        )
    )
    .withColumn(
        "failure_rate",
        F.when(
            F.col("transaction_count") > 0,
            F.col("failed_transactions")
            / F.col("transaction_count") * 100
        )
    )
    .withColumn(
        "fraud_rate",
        F.when(
            F.col("transaction_count") > 0,
            F.col("fraud_transactions")
            / F.col("transaction_count") * 100
        )
    )
)

write_gold_table(
    gold_daily_transaction_kpis,
    "gold_daily_transaction_kpis"
)

In [0]:
gold_monthly_transaction_kpis = (
    fact
    .groupBy("txn_year", "txn_month", "txn_year_month")
    .agg(
        F.count("*").alias("transaction_count"),

        F.countDistinct("customer_id").alias(
            "active_customers"
        ),

        F.countDistinct("account_id").alias(
            "active_accounts"
        ),

        F.sum("absolute_amount").alias(
            "transaction_amount"
        ),

        F.sum("successful_amount").alias(
            "successful_amount"
        ),

        F.sum("failed_amount").alias(
            "failed_amount"
        ),

        F.sum("fraud_amount").alias(
            "fraud_amount"
        ),

        F.sum(
            F.when(F.col("is_success"), 1).otherwise(0)
        ).alias("successful_transactions"),

        F.sum(
            F.when(F.col("is_failed"), 1).otherwise(0)
        ).alias("failed_transactions"),

        F.sum(
            F.when(F.col("is_fraud"), 1).otherwise(0)
        ).alias("fraud_transactions"),

        F.avg("absolute_amount").alias(
            "average_transaction_amount"
        ),

        F.sum("net_amount").alias(
            "net_transaction_amount"
        )
    )
    .withColumn(
        "success_rate",
        F.when(
            F.col("transaction_count") > 0,
            F.col("successful_transactions")
            / F.col("transaction_count") * 100
        )
    )
    .withColumn(
        "failure_rate",
        F.when(
            F.col("transaction_count") > 0,
            F.col("failed_transactions")
            / F.col("transaction_count") * 100
        )
    )
    .withColumn(
        "fraud_rate",
        F.when(
            F.col("transaction_count") > 0,
            F.col("fraud_transactions")
            / F.col("transaction_count") * 100
        )
    )
)

write_gold_table(
    gold_monthly_transaction_kpis,
    "gold_monthly_transaction_kpis"
)

In [0]:
gold_customer_kpis = (
    fact
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("transaction_count"),

        F.countDistinct("account_id").alias(
            "account_count"
        ),

        F.sum("absolute_amount").alias(
            "total_transaction_amount"
        ),

        F.sum("successful_amount").alias(
            "successful_amount"
        ),

        F.sum("failed_amount").alias(
            "failed_amount"
        ),

        F.sum("fraud_amount").alias(
            "fraud_amount"
        ),

        F.sum(
            F.when(F.col("is_success"), 1).otherwise(0)
        ).alias("successful_transactions"),

        F.sum(
            F.when(F.col("is_failed"), 1).otherwise(0)
        ).alias("failed_transactions"),

        F.sum(
            F.when(F.col("is_fraud"), 1).otherwise(0)
        ).alias("fraud_transactions"),

        F.avg("absolute_amount").alias(
            "average_transaction_amount"
        ),

        F.max("txn_timestamp").alias(
            "last_transaction_timestamp"
        ),

        F.min("txn_timestamp").alias(
            "first_transaction_timestamp"
        )
    )
    .withColumn(
        "success_rate",
        F.when(
            F.col("transaction_count") > 0,
            F.col("successful_transactions")
            / F.col("transaction_count") * 100
        )
    )
    .withColumn(
        "fraud_rate",
        F.when(
            F.col("transaction_count") > 0,
            F.col("fraud_transactions")
            / F.col("transaction_count") * 100
        )
    )
)

write_gold_table(
    gold_customer_kpis,
    "gold_customer_kpis"
)

In [0]:
gold_account_kpis = (
    fact
    .groupBy(
        "account_id",
        "customer_id",
        "account_type",
        "branch",
        "currency",
        "balance"
    )
    .agg(
        F.count("*").alias("transaction_count"),

        F.sum("absolute_amount").alias(
            "total_transaction_amount"
        ),

        F.sum("successful_amount").alias(
            "successful_amount"
        ),

        F.sum("failed_amount").alias(
            "failed_amount"
        ),

        F.sum("fraud_amount").alias(
            "fraud_amount"
        ),

        F.sum(
            F.when(F.col("is_success"), 1).otherwise(0)
        ).alias("successful_transactions"),

        F.sum(
            F.when(F.col("is_failed"), 1).otherwise(0)
        ).alias("failed_transactions"),

        F.sum(
            F.when(F.col("is_fraud"), 1).otherwise(0)
        ).alias("fraud_transactions"),

        F.avg("absolute_amount").alias(
            "average_transaction_amount"
        ),

        F.sum("net_amount").alias(
            "net_transaction_amount"
        ),

        F.max("txn_timestamp").alias(
            "last_transaction_timestamp"
        )
    )
)

write_gold_table(
    gold_account_kpis,
    "gold_account_kpis"
)

In [0]:
gold_branch_kpis = (
    fact
    .groupBy("branch")
    .agg(
        F.count("*").alias("transaction_count"),

        F.countDistinct("customer_id").alias(
            "active_customers"
        ),

        F.countDistinct("account_id").alias(
            "active_accounts"
        ),

        F.sum("absolute_amount").alias(
            "total_transaction_amount"
        ),

        F.sum("successful_amount").alias(
            "successful_amount"
        ),

        F.sum("failed_amount").alias(
            "failed_amount"
        ),

        F.sum("fraud_amount").alias(
            "fraud_amount"
        ),

        F.sum(
            F.when(F.col("is_success"), 1).otherwise(0)
        ).alias("successful_transactions"),

        F.sum(
            F.when(F.col("is_failed"), 1).otherwise(0)
        ).alias("failed_transactions"),

        F.sum(
            F.when(F.col("is_fraud"), 1).otherwise(0)
        ).alias("fraud_transactions"),

        F.avg("absolute_amount").alias(
            "average_transaction_amount"
        ),

        F.sum("net_amount").alias(
            "net_transaction_amount"
        )
    )
    .withColumn(
        "success_rate",
        F.when(
            F.col("transaction_count") > 0,
            F.col("successful_transactions")
            / F.col("transaction_count") * 100
        )
    )
    .withColumn(
        "fraud_rate",
        F.when(
            F.col("transaction_count") > 0,
            F.col("fraud_transactions")
            / F.col("transaction_count") * 100
        )
    )
)

write_gold_table(
    gold_branch_kpis,
    "gold_branch_kpis"
)

In [0]:
gold_fraud_kpis = (
    fact
    .groupBy("txn_date")
    .agg(
        F.count("*").alias("total_transactions"),

        F.sum(
            F.when(F.col("is_fraud"), 1).otherwise(0)
        ).alias("fraud_transactions"),

        F.sum("fraud_amount").alias(
            "fraud_amount"
        ),

        F.sum("absolute_amount").alias(
            "total_transaction_amount"
        )
    )
    .withColumn(
        "fraud_rate",
        F.when(
            F.col("total_transactions") > 0,
            F.col("fraud_transactions")
            / F.col("total_transactions") * 100
        )
    )
)

write_gold_table(
    gold_fraud_kpis,
    "gold_fraud_kpis"
)


gold_fraud_by_branch = (
    fact
    .groupBy("branch")
    .agg(
        F.count("*").alias("total_transactions"),

        F.sum(
            F.when(F.col("is_fraud"), 1).otherwise(0)
        ).alias("fraud_transactions"),

        F.sum("fraud_amount").alias(
            "fraud_amount"
        ),

        F.sum("absolute_amount").alias(
            "total_transaction_amount"
        )
    )
    .withColumn(
        "fraud_rate",
        F.when(
            F.col("total_transactions") > 0,
            F.col("fraud_transactions")
            / F.col("total_transactions") * 100
        )
    )
)

write_gold_table(
    gold_fraud_by_branch,
    "gold_fraud_by_branch"
)



gold_fraud_by_channel = (
    fact
    .groupBy("channel")
    .agg(
        F.count("*").alias("total_transactions"),

        F.sum(
            F.when(F.col("is_fraud"), 1).otherwise(0)
        ).alias("fraud_transactions"),

        F.sum("fraud_amount").alias(
            "fraud_amount"
        )
    )
    .withColumn(
        "fraud_rate",
        F.when(
            F.col("total_transactions") > 0,
            F.col("fraud_transactions")
            / F.col("total_transactions") * 100
        )
    )
)

write_gold_table(
    gold_fraud_by_channel,
    "gold_fraud_by_channel"
)

In [0]:

daily = spark.table(
    f"{GOLD_DATABASE}.gold_daily_transaction_kpis"
)

daily_window = Window.orderBy("txn_date")

gold_daily_trends = (
    daily
    .withColumn(
        "previous_day_transaction_amount",
        F.lag("transaction_amount").over(daily_window)
    )
    .withColumn(
        "previous_day_transaction_count",
        F.lag("transaction_count").over(daily_window)
    )
    .withColumn(
        "amount_change",
        F.col("transaction_amount")
        - F.col("previous_day_transaction_amount")
    )
    .withColumn(
        "transaction_count_change",
        F.col("transaction_count")
        - F.col("previous_day_transaction_count")
    )
    .withColumn(
        "amount_change_pct",
        F.when(
            F.col("previous_day_transaction_amount") != 0,
            (
                F.col("transaction_amount")
                - F.col("previous_day_transaction_amount")
            )
            / F.col("previous_day_transaction_amount") * 100
        )
    )
    .withColumn(
        "transaction_count_change_pct",
        F.when(
            F.col("previous_day_transaction_count") != 0,
            (
                F.col("transaction_count")
                - F.col("previous_day_transaction_count")
            )
            / F.col("previous_day_transaction_count") * 100
        )
    )
)

write_gold_table(
    gold_daily_trends,
    "gold_daily_trends"
)


monthly = spark.table(
    f"{GOLD_DATABASE}.gold_monthly_transaction_kpis"
)

monthly_window = Window.orderBy(
    "txn_year",
    "txn_month"
)

gold_monthly_trends = (
    monthly
    .withColumn(
        "previous_month_transaction_amount",
        F.lag("transaction_amount").over(monthly_window)
    )
    .withColumn(
        "previous_month_transaction_count",
        F.lag("transaction_count").over(monthly_window)
    )
    .withColumn(
        "amount_change",
        F.col("transaction_amount")
        - F.col("previous_month_transaction_amount")
    )
    .withColumn(
        "transaction_count_change",
        F.col("transaction_count")
        - F.col("previous_month_transaction_count")
    )
    .withColumn(
        "amount_change_pct",
        F.when(
            F.col("previous_month_transaction_amount") != 0,
            (
                F.col("transaction_amount")
                - F.col("previous_month_transaction_amount")
            )
            / F.col("previous_month_transaction_amount") * 100
        )
    )
    .withColumn(
        "transaction_count_change_pct",
        F.when(
            F.col("previous_month_transaction_count") != 0,
            (
                F.col("transaction_count")
                - F.col("previous_month_transaction_count")
            )
            / F.col("previous_month_transaction_count") * 100
        )
    )
)

write_gold_table(
    gold_monthly_trends,
    "gold_monthly_trends"
)

In [0]:
monthly_base = (
    fact
    .withColumn(
        "month_start",
        F.trunc(F.to_date("txn_timestamp"), "month")
    )
    .groupBy("month_start")
    .agg(
        F.count("*").alias("transaction_count"),

        F.countDistinct("customer_id").alias(
            "active_customers"
        ),

        F.countDistinct("account_id").alias(
            "active_accounts"
        ),

        F.sum("absolute_amount").alias(
            "transaction_amount"
        ),

        F.sum("fraud_amount").alias(
            "fraud_amount"
        ),

        F.sum(
            F.when(F.col("is_fraud"), 1).otherwise(0)
        ).alias("fraud_transactions")
    )
)

month_bounds = monthly_base.agg(
    F.min("month_start").alias("min_month"),
    F.max("month_start").alias("max_month")
).first()


if month_bounds["min_month"] is not None:

    month_calendar = spark.sql(
        f"""
        SELECT explode(
            sequence(
                DATE('{month_bounds["min_month"]}'),
                DATE('{month_bounds["max_month"]}'),
                INTERVAL 1 MONTH
            )
        ) AS month_start
        """
    )

    monthly_continuous = (
        month_calendar
        .join(
            monthly_base,
            on="month_start",
            how="left"
        )
        .fillna({
            "transaction_count": 0,
            "active_customers": 0,
            "active_accounts": 0,
            "transaction_amount": 0.0,
            "fraud_amount": 0.0,
            "fraud_transactions": 0
        })
        .withColumn(
            "year_month",
            F.date_format("month_start", "yyyy-MM")
        )
    )

    month_window = Window.orderBy("month_start")

    gold_period_over_period = (
        monthly_continuous
        .withColumn(
            "previous_month_amount",
            F.lag("transaction_amount").over(month_window)
        )
        .withColumn(
            "previous_month_count",
            F.lag("transaction_count").over(month_window)
        )
        .withColumn(
            "previous_month_fraud_amount",
            F.lag("fraud_amount").over(month_window)
        )
        .withColumn(
            "previous_month_fraud_count",
            F.lag("fraud_transactions").over(month_window)
        )
        .withColumn(
            "amount_change",
            F.col("transaction_amount")
            - F.col("previous_month_amount")
        )
        .withColumn(
            "transaction_count_change",
            F.col("transaction_count")
            - F.col("previous_month_count")
        )
        .withColumn(
            "fraud_amount_change",
            F.col("fraud_amount")
            - F.col("previous_month_fraud_amount")
        )
        .withColumn(
            "amount_change_pct",
            F.when(
                F.col("previous_month_amount") > 0,
                F.col("amount_change")
                / F.col("previous_month_amount") * 100
            )
        )
        .withColumn(
            "transaction_count_change_pct",
            F.when(
                F.col("previous_month_count") > 0,
                F.col("transaction_count_change")
                / F.col("previous_month_count") * 100
            )
        )
        .withColumn(
            "fraud_amount_change_pct",
            F.when(
                F.col("previous_month_fraud_amount") > 0,
                F.col("fraud_amount_change")
                / F.col("previous_month_fraud_amount") * 100
            )
        )
        .withColumn(
            "fraud_rate",
            F.when(
                F.col("transaction_count") > 0,
                F.col("fraud_transactions")
                / F.col("transaction_count") * 100
            )
        )
    )

    write_gold_table(
        gold_period_over_period,
        "gold_period_over_period"
    )

else:
    print("No transaction months found.")

In [0]:
daily_running = (
    fact
    .groupBy("txn_date")
    .agg(
        F.count("*").alias("transaction_count"),

        F.sum("absolute_amount").alias(
            "transaction_amount"
        ),

        F.sum("successful_amount").alias(
            "successful_amount"
        ),

        F.sum("fraud_amount").alias(
            "fraud_amount"
        )
    )
)

running_window = (
    Window
    .orderBy("txn_date")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

gold_running_transactions = (
    daily_running
    .withColumn(
        "running_transaction_count",
        F.sum("transaction_count").over(running_window)
    )
    .withColumn(
        "running_transaction_amount",
        F.sum("transaction_amount").over(running_window)
    )
    .withColumn(
        "running_successful_amount",
        F.sum("successful_amount").over(running_window)
    )
    .withColumn(
        "running_fraud_amount",
        F.sum("fraud_amount").over(running_window)
    )
)

write_gold_table(
    gold_running_transactions,
    "gold_running_transactions"
)

In [0]:
customer_dim = spark.table(
    f"{GOLD_DATABASE}.dim_customers"
)

customer_kpis = spark.table(
    f"{GOLD_DATABASE}.gold_customer_kpis"
)

gold_customer_mart = (
    customer_dim
    .join(
        customer_kpis,
        on="customer_id",
        how="left"
    )
    .fillna({
        "transaction_count": 0,
        "account_count": 0,
        "total_transaction_amount": 0.0,
        "successful_amount": 0.0,
        "failed_amount": 0.0,
        "fraud_amount": 0.0,
        "successful_transactions": 0,
        "failed_transactions": 0,
        "fraud_transactions": 0
    })
)

write_gold_table(
    gold_customer_mart,
    "gold_customer_mart"
)

In [0]:
account_dim = spark.table(
    f"{GOLD_DATABASE}.dim_accounts"
)

account_kpis = spark.table(
    f"{GOLD_DATABASE}.gold_account_kpis"
)

gold_account_mart = (
    account_dim
    .join(
        account_kpis,
        on=[
            "account_id",
            "customer_id",
            "account_type",
            "branch",
            "currency",
            "balance"
        ],
        how="left"
    )
    .fillna({
        "transaction_count": 0,
        "total_transaction_amount": 0.0,
        "successful_amount": 0.0,
        "failed_amount": 0.0,
        "fraud_amount": 0.0,
        "successful_transactions": 0,
        "failed_transactions": 0,
        "fraud_transactions": 0
    })
)

write_gold_table(
    gold_account_mart,
    "gold_account_mart"
)

In [0]:
branch_kpis = spark.table(
    f"{GOLD_DATABASE}.gold_branch_kpis"
)

gold_branch_mart = branch_kpis.select(
    "branch",
    "transaction_count",
    "active_customers",
    "active_accounts",
    "total_transaction_amount",
    "successful_amount",
    "failed_amount",
    "fraud_amount",
    "successful_transactions",
    "failed_transactions",
    "fraud_transactions",
    "average_transaction_amount",
    "net_transaction_amount",
    "success_rate",
    "fraud_rate"
)

write_gold_table(
    gold_branch_mart,
    "gold_branch_mart"
)

In [0]:
gold_transaction_mart = (
    fact
    .select(
        "txn_id",
        "txn_date",
        "date_key",
        "txn_timestamp",
        "txn_year",
        "txn_month",
        "txn_year_month",
        "txn_hour",
        "txn_day_of_week",
        "customer_id",
        "account_id",
        "account_type",
        "branch",
        "currency",
        "txn_type",
        "txn_type_normalized",
        "channel",
        "merchant_category",
        "amount",
        "absolute_amount",
        "net_amount",
        "status",
        "status_normalized",
        "is_success",
        "is_failed",
        "is_pending",
        "is_fraud",
        "fraud_amount"
    )
)

write_gold_table(
    gold_transaction_mart,
    "gold_transaction_mart"
)

In [0]:
RUN_END_TIME = datetime.utcnow()

try:

    gold_table_names = [
        "dim_customers",
        "dim_accounts",
        "dim_date",
        "fact_transactions",
        "gold_transaction_kpis",
        "gold_daily_transaction_kpis",
        "gold_monthly_transaction_kpis",
        "gold_customer_kpis",
        "gold_account_kpis",
        "gold_branch_kpis",
        "gold_fraud_kpis",
        "gold_fraud_by_branch",
        "gold_fraud_by_channel",
        "gold_daily_trends",
        "gold_monthly_trends",
        "gold_period_over_period",
        "gold_running_transactions",
        "gold_customer_mart",
        "gold_account_mart",
        "gold_branch_mart",
        "gold_transaction_mart"
    ]

    total_gold_records = 0

    for table_name in gold_table_names:

        full_table_name = f"{GOLD_DATABASE}.{table_name}"

        if spark.catalog.tableExists(full_table_name):

            total_gold_records += (
                spark.table(full_table_name).count()
            )

    append_gold_log(
        table_name="__GOLD_RUN__",
        start_time=RUN_START_TIME,
        end_time=RUN_END_TIME,
        status="SUCCESS",
        records_processed=total_gold_records,
        error_message=None
    )

    print("===================================")
    print("GOLD LAYER COMPLETED")
    print("===================================")
    print("Run ID:", RUN_ID)
    print("Total records across Gold outputs:", total_gold_records)
    print("Gold schema:", GOLD_DATABASE)
    print("Log table:", LOG_TABLE)

except Exception as e:

    try:
        append_gold_log(
            table_name="__GOLD_RUN__",
            start_time=RUN_START_TIME,
            end_time=datetime.utcnow(),
            status="FAILED",
            records_processed=0,
            error_message=str(e)
        )
    except Exception as log_error:
        print("Could not write run-level log:", str(log_error))

    raise

In [0]:
gold_tables = [
    "dim_customers",
    "dim_accounts",
    "dim_date",
    "fact_transactions",
    "gold_transaction_kpis",
    "gold_daily_transaction_kpis",
    "gold_monthly_transaction_kpis",
    "gold_customer_kpis",
    "gold_account_kpis",
    "gold_branch_kpis",
    "gold_fraud_kpis",
    "gold_fraud_by_branch",
    "gold_fraud_by_channel",
    "gold_daily_trends",
    "gold_monthly_trends",
    "gold_period_over_period",
    "gold_running_transactions",
    "gold_customer_mart",
    "gold_account_mart",
    "gold_branch_mart",
    "gold_transaction_mart"
]

for table_name in gold_tables:

    full_table_name = f"{GOLD_DATABASE}.{table_name}"

    if spark.catalog.tableExists(full_table_name):

        print(f"\n===== {full_table_name} =====")

        display(
            spark.table(full_table_name).limit(10)
        )

In [0]:
gold_logs = spark.table(LOG_TABLE)

display(
    gold_logs
    .filter(F.col("run_id") == RUN_ID)
    .orderBy("start_time")
)

In [0]:
%sql
Select * from stream_bank_processing.gold.gold_period_over_period